# Lab 4 — Notebook 02: Classification — Recognizing Handwritten Digits

**Course:** Artificial Intelligence with Python  
**Duration:** 55 minutes (core route)  
**Prerequisites:** Labs 1–3 (Python, data preparation, models, loss and generalization)  
**Files:** `data/digits.csv` (included in this lab folder)  
**Libraries:** NumPy, Pandas, Matplotlib, scikit-learn

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain **classification** and distinguish it from regression.
2. Load a labeled dataset from CSV and visualize 8 × 8 pixel images.
3. Create reproducible **stratified training, validation and test splits**.
4. Train and use **k-nearest neighbors**, **logistic regression**, and **decision tree**, and **random forest** classifiers.
5. Evaluate classifiers on **validation data** using accuracy, a confusion matrix, precision, recall, and F1-score.
6. Examine mistakes without inspecting the reserved test set.

**Working convention:** Run example cells from top to bottom. Exercise cells contain `TODO` prompts; add code and short written answers. Each notebook is independent.

**Dataset:** The handwritten digits dataset bundled with scikit-learn (`sklearn.datasets.load_digits`), exported to CSV. It contains **1,797** labeled samples, each stored as 64 grayscale pixel intensities (integers from **0 to 16**) representing an **8 × 8 image**. The label `target` is one of the digits **0–9**. No internet connection is needed.

**Experimental rule:** Notebook 02 uses **training and validation data only** to investigate classifiers. Notebook 03 independently recreates the *identical* split (same `random_state=42` and stratification) and reveals the reserved test set only for the final evaluation. **Do not view or score test observations in this notebook.**


**Timing:** Section budgets include core exercises: 4 + 5 + 5 + 9 + 13 + 12 + 5 + 2 = **55 minutes**. The extra prediction exercise is optional. Examples run independently of TODO answers. Restart and Run All when finished.


## 1. Load and explore handwritten digits (4 min)

In **regression**, the output is a number, such as monthly apartment rent. In **classification**, the output is a **category**. Here we predict which handwritten digit (0–9) appears in an image.

The CSV stores an image as one row of **64 numerical features** plus a `target` label. Load it with Pandas and check whether it contains the expected columns and values.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

csv_path = Path("data/digits.csv")
if not csv_path.is_file():
    raise FileNotFoundError(
        "Missing data/digits.csv. Open JupyterLab from the course repository root "
        "and work inside lab04, or place the CSV in lab04/data/."
    )

df = pd.read_csv(csv_path)
pixel_columns = [f"pixel_{i}" for i in range(64)]
assert df.shape == (1797, 65), "Unexpected dataset shape; check digits.csv"
assert set(pixel_columns + ["target"]) == set(df.columns)
assert df[pixel_columns].min().min() >= 0
assert df[pixel_columns].max().max() <= 16

print("Dataset shape:", df.shape)

print("Missing values:", int(df.isna().sum().sum()))


## 2. Separate features and labels; split the data (5 min, including reflection)

A classifier learns from examples for which the input features (`X`) and correct labels (`y`) are known.

- `X`: 64 pixels for each image (**exclude `target`**).
- `y`: the correct digit (0–9).
- **Training set:** used to fit classifiers.
- **Validation set:** used to compare classifiers, compute metrics, and investigate errors.
- **Test set:** reserved for **Notebook 03** and not inspected here.

First reserve 20% for the final test set. Then reserve 20% of the remaining 80% for validation. The resulting proportions are approximately **64% training / 16% validation / 20% test**. We use `stratify` to preserve label proportions and a fixed random state for reproducibility. **Notebook 03 uses exactly this split.**


In [ ]:
from sklearn.model_selection import train_test_split

X = df[pixel_columns]
y = df["target"]

# Reserve the test set for Notebook 03. Do not inspect or score it here.
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_development, y_development,
    test_size=0.20, random_state=42, stratify=y_development
)

print("Training:", X_train.shape, "Validation:", X_val.shape)
print("Reserved for final evaluation:", len(X_test), "images (not used here)")

proportions = pd.DataFrame({
    "Training": y_train.value_counts(normalize=True).sort_index(),
    "Validation": y_val.value_counts(normalize=True).sort_index(),
})
display(proportions.round(3))
assert X_train.index.intersection(X_val.index).empty
assert X_development.index.intersection(X_test.index).empty
assert X_train.index.intersection(X_test.index).empty


### Exercise 2 — Explain stratification

1. How many images are in the **validation** set? How many are reserved for final testing?
2. Compare the fraction of digit **0** in the training and validation sets.
3. Why is the final test set kept untouched while we compare classifiers?

**Your answer:** *TODO: Write 2–3 sentences.*

### Core — How can a table contain images? (5 min, including Exercise 1)

Each row contains 64 pixels that can be reshaped to an 8 × 8 image. Higher values appear darker with `gray_r`. We display **training samples only**, keeping final test images unseen.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    image = X_train.iloc[i].to_numpy(dtype=float).reshape(8, 8)
    ax.imshow(image, cmap="gray_r", vmin=0, vmax=16, interpolation="nearest")
    ax.set_title(f"Label: {y_train.iloc[i]}")
    ax.axis("off")
plt.suptitle("Ten training examples")
plt.tight_layout()
plt.show()


### Core exercise 1 — Inspect a training example

Count the training examples labeled 7. Select one using `X_train.loc[y_train == 7]`, reshape its pixels, and display it. Explain why the pixels are features and the digit is the target.


In [ ]:
# TODO: count digit 7 in y_train
# TODO: display one corresponding image from X_train


**Your explanation:** *TODO: Explain the difference between a pixel feature and the target label.*

## 3. Train your first classifier: k-nearest neighbors (9 min, including baseline)

**k-nearest neighbors (k-NN)** predicts a new image's class by looking at the labels of its `k` most similar training images. Here, similarity is measured using Euclidean distance between the 64 pixel values.

Use a scikit-learn **Pipeline** to scale features first, then train the classifier. Scaling is learned from the **training set only**, avoiding data leakage.

> Unlike regression, `.predict()` now returns a **digit label**, not a continuous value.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn_model = Pipeline([
    ("scale", StandardScaler()),
    ("classifier", KNeighborsClassifier(n_neighbors=5))
])
knn_model.fit(X_train, y_train)

knn_predictions = knn_model.predict(X_val)
knn_accuracy = accuracy_score(y_val, knn_predictions)
print(f"k-NN validation accuracy: {knn_accuracy:.3f}")
print("First 10 predictions:", knn_predictions[:10])
print("First 10 true labels:", y_val.iloc[:10].to_numpy())

**Interpreting accuracy:** An accuracy of 0.95 means 95% of the tested images were classified correctly. Accuracy alone does not tell us which digits were confused.

We should also compare our trained classifier against a **simple baseline** that always predicts the most common training label.

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_val, baseline.predict(X_val))
print(f"Majority-class baseline validation accuracy: {baseline_accuracy:.3f}")
print(f"k-NN validation accuracy:                    {knn_accuracy:.3f}")

### Practice — Make another prediction (optional, 3 min)

Predict the first five validation images and compare their labels. Use the already fitted `knn_model`.


In [ ]:
# TODO: predict labels for X_val.iloc[:5]
# TODO: compare them with y_val.iloc[:5]

**Your explanation:** *TODO: Why evaluate on a validation set instead of training data?*

## 4. Compare four model families (13 min, including Exercise 4)

**Core.** Use the same training and validation observations for all models.

- **k-NN:** uses nearby training examples to vote.
- **Logistic regression:** a classifier despite its name. It learns linear decision boundaries in feature space and provides class probabilities. It does not perform ordinary numeric-target regression.
- **Decision tree:** learns a sequence of feature-based questions.
- **Random forest:** combines many trees trained with bootstrap samples and random feature subsets at splits. Combining diverse trees can reduce the instability of a single tree; improvement is not guaranteed.

Scaling belongs in the k-NN and logistic-regression pipelines. Trees do not need scaling for these pixel features. The forest uses 100 trees and one worker to remain modest on shared machines. Compare fixed configurations now; Notebook 03 will shortlist candidates and use cross-validation.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

models = {
    "k-NN (k=5)": knn_model,
    "Logistic regression": Pipeline([
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Decision tree": DecisionTreeClassifier(max_depth=10, random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1),
}

# k-NN is already trained; fit the other models on the same training set.
for name, model in models.items():
    if name != "k-NN (k=5)":
        model.fit(X_train, y_train)

results = []
for name, model in models.items():
    train_accuracy = accuracy_score(y_train, model.predict(X_train))
    validation_accuracy = accuracy_score(y_val, model.predict(X_val))
    results.append({
        "Model": name,
        "Train accuracy": train_accuracy,
        "Validation accuracy": validation_accuracy,
    })

results_df = pd.DataFrame(results)
display(results_df.round(3))


### Core exercise 4 — Does the forest help?

1. Which of the four fixed configurations has the highest validation accuracy?
2. Compare the decision tree and random forest: calculate each training–validation gap.
3. Does the forest improve validation accuracy here? Does this guarantee it wins on another dataset?

**Your explanation:** TODO — Use numbers from the table and distinguish training fit from generalization.


## 5. Understand mistakes: confusion matrix and class metrics (12 min, including Exercise 5)

A **confusion matrix** counts how often each true digit is predicted as each possible digit.

- **Rows:** true class.
- **Columns:** predicted class.
- A strong classifier has most counts on the **main diagonal**.

We'll inspect k-NN, our first classifier, and look at the images it gets wrong.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(
    y_val, knn_predictions,
    labels=list(range(10)),
    cmap="Blues", values_format="d", ax=ax
)
ax.set_title("k-NN confusion matrix — validation data")
plt.tight_layout()
plt.show()

**Beyond accuracy:** For each digit, **precision** describes how frequently predictions of that digit are correct; **recall** describes how frequently actual examples of that digit are found. **F1-score** balances precision and recall. The report below also shows the number of **validation** samples in each class (`support`).

In [ ]:
print(classification_report(y_val, knn_predictions, digits=3, zero_division=0))

An incorrectly classified image is especially informative. The next cell displays a few errors with their true and predicted labels. It will show a helpful message if none were misclassified.

In [ ]:
wrong_positions = np.flatnonzero(
    knn_predictions != y_val.to_numpy()
)
print("Misclassified validation images:", len(wrong_positions), "out of", len(y_val))

if len(wrong_positions) > 0:
    count = min(6, len(wrong_positions))
    fig, axes = plt.subplots(1, count, figsize=(count * 2, 2.4))
    axes = np.atleast_1d(axes)
    for ax, pos in zip(axes, wrong_positions[:count]):
        pixels = X_val.iloc[pos].to_numpy().reshape(8, 8)
        ax.imshow(pixels, cmap="gray_r", vmin=0, vmax=16)
        ax.set_title(f"True: {y_val.iloc[pos]}\nPred: {knn_predictions[pos]}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No mistakes found in this validation split.")

### Exercise 5 — Investigate an error

1. Find the **largest off-diagonal count** in the confusion matrix. Which two digits are involved?
2. Look at a misclassified digit above. Can you explain why the handwriting might be difficult to recognize?
3. Why might high overall accuracy still hide poor performance on an individual class?

**Hint:** `from sklearn.metrics import confusion_matrix` returns the numerical matrix. To ignore its diagonal, you can set those entries to zero with `np.fill_diagonal(matrix, 0)`.

In [ ]:
# TODO: create a confusion matrix for y_val and knn_predictions
# TODO: ignore the diagonal and find the largest off-diagonal entry
# TODO: identify its true digit (row) and predicted digit (column)

**Your observations:** *TODO: Write 2–3 sentences about the errors you found.*

## 6. Core mini-challenge — Compare a failure (5 min)

Choose one of k-NN's misclassified **validation** images displayed above. Compare the predictions of all four models on that same image. If k-NN has no errors, choose any validation image.

1. Show the image, true label, and four predictions.
2. Do the tree and forest agree? Does either correct k-NN's mistake?
3. Explain why one corrected example cannot establish overall superiority.

Use the already trained models. Do not inspect test images or tune settings.


In [ ]:
# TODO: choose pos from wrong_positions; use pos = 0 if there are no mistakes
# TODO: display X_val.iloc[pos].to_numpy().reshape(8, 8)
# TODO: loop over models.items() and predict X_val.iloc[[pos]]
# TODO: compare predictions with y_val.iloc[pos] and explain the result


### Core wrap-up (2 min)

- Why can a forest generalize better than a single tree?
- Why is logistic regression used for classification here?
- Why might overall accuracy conceal weakness on one digit?
- Why must preprocessing be fitted inside each training partition?

**Next:** Notebook 03 recreates the same split, explores a few k-NN/tree settings, includes logistic regression and random forest in the candidate comparison, and shortlists two candidates. Five-fold cross-validation on development data supports final selection. Only then is the reserved test set evaluated.

**Deliverable:** One validation comparison table, the forest-versus-tree explanation, and one inspected failure.
